# Post-batch steps for sessions the batch has finished

Run this in its own kernel while `batch_and_exports.ipynb` (or the batch in Terminal) keeps going. For the sessions in `SESSIONS`, it does the steps that normally follow the batch, but only for runs **the running batch has finished**:

1. **Contrast exports:** green and red mean images per run (raw TIFFs, contrast PNGs, overlay), with the `[exports]` contrast.
2. **Cellpose extraction:** `run-XX/<cellpose_seg>`, drawn on `run-XX/<cellpose_image>`, aligned onto the run's new mean image and extracted, with the same checks as the batch notebook.
3. **Alignment review:** a table, and `mask_alignment_ses-<N>_<variant>.pdf` in the subject folder, titled with how many of the session's runs it covers.

"Finished" uses the cutoff the running batch was launched with (its frozen `batch_config_*.toml`), not `pipeline.toml`, so editing the config mid-batch cannot make an unfinished run look ready. Every run folder is listed with its state: only `ready` runs are processed. `awaiting rerun` still holds the old registration and `running` is half-written; the batch will get to them. `failed`, `interrupted` and `no batch record` will not change on their own, so look at those.

Re-running is safe. Extractions from the same masks are kept, and exports are kept unless `[exports] overwrite = true`. **Stop using this notebook before the batch notebook goes on to its own export and extraction cells**, so the two never write to the same run at once.

In [ ]:
import logging
from pathlib import Path

from mini2p_preprocessing.batch import launch_cutoff, load_batch_settings, run_state
from mini2p_preprocessing.cellpose_batch import (
    ATTENTION,
    extract_run,
    format_summary,
    summary_row,
    write_mask_alignment_pdf,
)
from mini2p_preprocessing.config import VARIANTS, find_registered_plane, load_settings, variant_label
from mini2p_preprocessing.exports import SESSION_PATTERN, export_mean_images, load_export_settings
from mini2p_preprocessing.progress import ProgressLog

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)
CONFIG = Path("../config/pipeline.toml").resolve()
SESSIONS = [216]  # sessions to process; add more as the batch finishes them

batch = load_batch_settings(CONFIG)
exports = load_export_settings(CONFIG)
settings = load_settings(CONFIG)
cutoff = launch_cutoff(batch.derivatives_subject_dir)  # what the running batch uses
print("Ready = completed by the batch", f"since {cutoff}" if cutoff else "(no rerun cutoff)")

## Which runs are ready

In [ ]:
ready, total = [], {}
for session in SESSIONS:
    session_dirs = [
        d for d in Path(exports.subject_dir).glob("ses-*")
        if (m := SESSION_PATTERN.match(d.name)) and int(m.group(1)) == session
    ]
    if len(session_dirs) != 1:
        print(f"ses-{session}: expected one session folder, found {len(session_dirs)}")
        continue
    for run_dir in sorted((session_dirs[0] / "funcimg").glob("run-*")):
        for variant in VARIANTS:  # every variant folder the run has
            if not (run_dir / variant).is_dir():
                continue
            state = run_state(run_dir / variant, cutoff)
            total[(session, variant)] = total.get((session, variant), 0) + 1
            print(f"ses-{session}  {run_dir.name}  {variant:<48}  {state}")
            if state == "ready":
                ready.append((session, run_dir, variant))
print(f"{len(ready)} of {sum(total.values())} run folders ready")

## Exports, extraction and alignment review

In [ ]:
rows = {}
progress = ProgressLog("Post-batch", len(ready), print)
for session, run_dir, variant in ready:
    plane_dir = find_registered_plane(run_dir / variant, exports.alignment)
    if plane_dir is None:
        print(f"ses-{session} {run_dir.name} {variant}: no {exports.alignment} registration")
        continue
    states = export_mean_images(plane_dir, exports.contrast, exports.overwrite)
    print(f"ses-{session} {run_dir.name} {variant} exports: {sorted(set(states.values()))}")
    row = summary_row(session, run_dir.name, variant, plane_dir)
    rows.setdefault((session, variant), []).append(extract_run(
        plane_dir, run_dir, run_dir / batch.cellpose_seg, run_dir / batch.cellpose_image,
        settings, False, row,
    ))
    progress.step_done()

for (session, variant), session_rows in sorted(rows.items()):
    covered = f"{len(session_rows)} of {total[(session, variant)]} runs"
    print(f"\nses-{session} {variant} ({covered})")
    print(format_summary(session_rows))
    for r in session_rows:
        if r["status"] in ATTENTION:
            print(f"  NEEDS ATTENTION {r['run']}: {r['status']}")
    pdf = exports.subject_dir / f"mask_alignment_ses-{session}_{variant_label(variant)}.pdf"
    write_mask_alignment_pdf(session_rows, pdf, exports.contrast,
                             f"ses-{session} {variant}: {covered}",
                             overwrite=True)  # a report rebuilt from the records
    print("Saved", pdf)